# Lab 08 Prelab Walkthrough — First-Order Response and Log-Linearization

Work through this notebook **before** your Lab 08 session, running every cell
and reading every explanation. It teaches this lab's *new* skills on a
simulated thermocouple — no hardware needed. The **CHECKPOINT** boxes ask
for specific values you will report in the **Prelab quiz on Canvas**.

Skills covered:

1. Simulating a **first-order step response** (`np.exp`)
2. The **63.2% method** on noisy data
3. **Log-transform linearization**: `np.log` + `polyfit` recover tau from all the data
4. Why the **fit range** (the Gamma mask) matters
5. **Gap differencing** (`T[10:] - T[:-10]`) for robust step detection

Run the setup cell first:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 10

## 1. A simulated plunge

A first-order sensor stepped from T0 to Tss follows
$T(t) = T_{ss} + (T_0 - T_{ss})e^{-t/\tau}$. Here is a pretend thermocouple
with tau = 60 ms, sampled at 1000 Hz, with realistic noise (seeded — everyone
gets identical numbers):

In [ ]:
rng = np.random.default_rng(42)

TAU  = 0.060                     # s  (the truth we'll try to recover)
T0, Tss = 22.0, 97.0             # C
fs   = 1000                      # Hz
t    = np.arange(0, 0.6, 1/fs)   # 0.6 s of data, step at t = 0

T = Tss + (T0 - Tss) * np.exp(-t / TAU)
T = T + rng.normal(0, 0.2, t.size)          # 0.2 C sensor noise

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(t, T, 'r.', markersize=3)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature (°C)')
ax.grid(linestyle='--', linewidth=0.5)
plt.show()

## 2. The 63.2% method

At exactly $t = \tau$, a first-order response has covered 63.2% of its
step. So: compute the 63.2% level, find the first crossing:

In [ ]:
T63    = T0 + 0.632 * (Tss - T0)
tau_63 = t[np.argmax(T >= T63)]

print(f"63.2% level = {T63:.1f} C")
print(f"tau (63.2% method) = {tau_63*1000:.1f} ms   (truth: {TAU*1000:.0f} ms)")

Close — but this estimate hangs on a *single* noisy sample crossing a
threshold. Run the first cell again with a different seed someday and watch
it wander.

> **CHECKPOINT 1:** Report the 63.2% level (°C, 1 decimal) and the tau the
> method returns (ms, 1 decimal).

## 3. Log-linearization: the error fraction

The remaining "distance to go", normalized, is the error fraction:

$$\Gamma(t) = \frac{T_{ss} - T(t)}{T_{ss} - T_0} = e^{-t/\tau}
\quad\Rightarrow\quad \ln\Gamma = -t/\tau$$

One `np.log` and the exponential becomes a straight line — which `polyfit`
fits using EVERY point:

In [ ]:
Gamma = (Tss - T) / (Tss - T0)

valid = (Gamma > 0.02) & (Gamma < 0.95)     # more on this mask below
t_v      = t[valid]
ln_Gamma = np.log(Gamma[valid])

coeffs = np.polyfit(t_v, ln_Gamma, 1)
tau_ef = -1.0 / coeffs[0]

print(f"slope = {coeffs[0]:.2f} 1/s")
print(f"tau (error fraction) = {tau_ef*1000:.1f} ms   (truth: {TAU*1000:.0f} ms)")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(t[Gamma > 0], np.log(Gamma[Gamma > 0]), 'r.', markersize=3,
        label='all data')
ax.plot(t_v, np.polyval(coeffs, t_v), 'b-', linewidth=2, label='fit (masked)')
ax.set_xlabel('Time (s)')
ax.set_ylabel(r'$\ln \Gamma$')
ax.legend()
ax.grid(linestyle='--', linewidth=0.5)
plt.show()

Look at the tail of the plot: past ~0.3 s the points spray wildly.
There, $T_{ss} - T(t)$ is smaller than the noise — you are taking the log of
noise, and the log of a small noisy number swings between huge negatives and
(for negative values) doesn't exist at all. The mask `Gamma > 0.02` fences
that region out of the fit.

> **CHECKPOINT 2:** Report tau from the error-fraction method (ms, 1
> decimal).

## 4. The mask matters — measure it

Refit with a recklessly wide mask and compare:

In [ ]:
for lo in [0.02, 0.005, 0.001]:
    m = (Gamma > lo) & (Gamma < 0.95)
    cf = np.polyfit(t[m], np.log(Gamma[m]), 1)
    print(f"mask Gamma > {lo:<6}: tau = {-1000/cf[0]:6.1f} ms  (N = {m.sum()})")

Widening the mask *adds* points but they are log-of-noise points, and
they drag the slope around. Fitting only what the model describes — same
discipline as Lab 05's saturation mask — beats fitting more data.

> **CHECKPOINT 3:** Report tau (ms, 1 decimal) for the `Gamma > 0.001` mask.

## 5. Finding the step: gap differencing

In lab the plunge happens at an unknown moment mid-record, and you must find
it. Single-sample differences drown in noise; the robust tool is the rise
across a wider gap (Lab 06's shifted slices):

In [ ]:
# a record where the step happens at 1.500 s
t2 = np.arange(0, 3.0, 1/fs)
T2 = np.where(t2 < 1.5, T0, Tss + (T0 - Tss) * np.exp(-(t2 - 1.5) / TAU))
T2 = T2 + rng.normal(0, 0.2, t2.size)

diff1  = np.diff(T2)                 # single-sample rise
rise10 = T2[10:] - T2[:-10]          # rise across 10 samples (10 ms)

print(f"single-sample rise: noise sigma ~ {diff1[:1000].std():.2f} C, "
      f"max in quiet region = {diff1[:1000].max():.2f} C")
print(f"10-sample rise:     noise sigma ~ {rise10[:1000].std():.2f} C, "
      f"but during the step it reaches {rise10.max():.1f} C")

idx = np.argmax(rise10 > 5.0)
print(f"\ndetected step near t = {t2[idx]:.3f} s (true: 1.500 s)")

The gap difference gives the step ~10 ms to build a huge signature
(tens of °C) while noise stays put — so a 5 °C threshold is untouchable by
noise yet trivially exceeded by the plunge. In lab, a refine step then
pins t = 0 to the exact takeoff sample.

> **CHECKPOINT 4:** Report the detected step time from the cell above
> (seconds, 3 decimals).

## Summary: where each skill is used in lab

| Skill | Where you will use it |
|---|---|
| `np.exp` step model | the model overlay on your step-response figure |
| 63.2% method | Part-5, all three runs |
| `np.log` + `polyfit` linearization | Part-6, the error-fraction fit |
| the Gamma mask | Part-6 (and logbook Q11 moves it) |
| gap differencing `T[10:] - T[:-10]` | the plunge detector in `analyze_run` |

See you in lab — the water will be hot, the transient will be over in a
blink, and your DAQ will catch it anyway.